# Custom miner model — Colab starter

## Disclaimer (read before changing anything)

- **Do not** drop, reorder, or rename feature columns relative to `feature_contract.json`. The miner expects the **exact same feature order** at inference.
- **Do not** change the meaning of the target column or row alignment (no manual shifting of labels vs features).
- **Only** change the model, training hyperparameters, and training code paths you understand.
- **Security**: `joblib` can execute arbitrary code when loading untrusted files. Only train/load artifacts you created.

Workflow: load CSV → align features → temporal split → fit → save `model_custom.joblib` or `model_custom.keras` → upload that file into the same plugin folder on your VM.

In [ ]:
# Optional (Colab): install TensorFlow for the Keras example
# !pip install -q tensorflow scikit-learn pandas numpy joblib

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PLUGIN_DIR = Path('.')  # In Colab: Path('/content/drive/MyDrive/.../your_plugin_folder')

contract = json.loads((PLUGIN_DIR / 'feature_contract.json').read_text())
FEATURES = contract['features']
Y_COL = contract['target_y_column']
VAL_FRAC = float(contract.get('validation_split', 0.2))

df = pd.read_csv(PLUGIN_DIR / 'training_dataset_full.csv')
df = df.sort_values(contract['timestamp_column']).reset_index(drop=True)
missing = [c for c in FEATURES if c not in df.columns]
assert not missing, f'Missing feature columns: {missing[:10]}'
X = df[FEATURES].astype(float).to_numpy()
y = df[Y_COL].astype(float).to_numpy()
n = len(df)
split = int(n * (1.0 - VAL_FRAC))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
X_train.shape, X_val.shape

## Example A — scikit-learn → `model_custom.joblib`

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

model = HistGradientBoostingRegressor(random_state=contract.get('random_state', 42))
model.fit(X_train, y_train)
out = PLUGIN_DIR / 'model_custom.joblib'
joblib.dump(model, out)
print('Saved', out)

## Example B — Keras dense regression → `model_custom.keras`

Use a **single vector input** `(batch, n_features)` so the miner probe matches. For sequence models you must fix `n_steps` in the Input layer and train with matching windows; the miner will build live windows only if your `model_params.yaml` feature settings require history (advanced).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

n_features = X_train.shape[1]
inp = keras.Input(shape=(n_features,))
x = layers.Dense(64, activation='relu')(inp)
x = layers.Dense(32, activation='relu')(x)
out = layers.Dense(1)(x)
kmodel = keras.Model(inp, out)
kmodel.compile(optimizer='adam', loss='mse')
kmodel.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=256, verbose=1)
kpath = PLUGIN_DIR / 'model_custom.keras'
kmodel.save(kpath)
print('Saved', kpath)